In [1]:
!pip install groq python-dotenv

In [2]:
# configuration and experts
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables 
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# UPDATED MODEL: Using the latest versatile Llama 3.3 model
MODEL = "llama-3.3-70b-versatile"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a Senior Software Engineer. Provide rigorous, code-focused, and precise technical solutions. Wrap code in Markdown blocks.",
        "temperature": 0.3
    },
    "billing": {
        "system_prompt": "You are a Billing Specialist. You are empathetic, follow strict company financial policies, and provide clear steps for refunds.",
        "temperature": 0.1
    },
    "general": {
        "system_prompt": "You are a friendly General Support Assistant. Answer briefly and politely.",
        "temperature": 0.7
    }
}

In [3]:
# router and tools
def route_prompt(user_input):
    """Classifies the user input into a category."""
    routing_instructions = (
        "Classify this text into EXACTLY one category: [technical, billing, general, tools]. "
        "If the user asks for the price of Bitcoin, return 'tools'. "
        "Return ONLY the word, nothing else."
    )
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": routing_instructions},
                {"role": "user", "content": user_input}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        return f"error: {str(e)}"

def get_bitcoin_price():
    """Mock function for the Bonus Challenge."""
    return "$92,450.12 USD"

In [4]:
# orchestrator
def process_request(user_input):
    # 1. Routing logic
    category = route_prompt(user_input)
    print(f"DEBUG: Router selected -> {category}")

    # 2. Handle Tool Use (Bonus)
    if "tools" in category:
        return f"The current price of Bitcoin is {get_bitcoin_price()}."

    # 3. Handle Experts
    expert_config = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=expert_config["temperature"]
    )
    
    return response.choices[0].message.content

In [6]:
# testing
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "What is the current price of Bitcoin?",
    "How do I change my password?"
]
    
for i, q in enumerate(test_queries, 1):
    print(f"\n--- STARTING QUERY {i} ---")
    print(f"USER INPUT: {q}")
    
    response = process_request(q)
    
    print(f"FINAL OUTPUT: {response}")
    print(f"--- END OF QUERY {i} ---\n")


--- STARTING QUERY 1 ---
USER INPUT: My python script is throwing an IndexError on line 5.
DEBUG: Router selected -> technical
FINAL OUTPUT: To provide a precise solution, I'll need to see the code. However, I can guide you through a general approach to debugging an `IndexError` in Python.

An `IndexError` typically occurs when you're trying to access an element in a list or other sequence that doesn't exist (e.g., an index that is out of range).

Here's a step-by-step approach to debugging the issue:

1. **Verify the Line of Code**: Check the line of code where the error is occurring (line 5 in your case). Ensure that you're not trying to access an index that doesn't exist.

2. **Check Sequence Length**: Before accessing an element, verify the length of the sequence (list, tuple, etc.) to ensure the index you're trying to access is within bounds.

3. **Use Defensive Programming**: Consider using defensive programming techniques, such as checking the length of the sequence before tryi